# Colab Runbook — Vietnamese ABSA

Chọn **Runtime → Change runtime type → GPU**. Upload và giải nén `absa_colab_backend.zip` vào `/content`, rồi chạy từng cell. Không lấy test để điều chỉnh tham số.

**Python 3.13:** `requirements.txt` pins `pyjnius>=1.7,<1.8` because prior versions lack Python 3.13 support. If pip reports `metadata-generation-failed`, stop and share the *first package name* in full error log; green tests do not mean dependencies installed.


In [ ]:
%cd /content/absa_colab
# Update installer and install dependencies. Do not hide install errors.
%pip install -U pip setuptools wheel
%pip install --prefer-binary -r requirements.txt
# Verify packages BEFORE running tests or downloading the dataset.
!python -m pip check
!python -c "import jnius, py_vncorenlp, transformers, sklearn, seqeval; print('Dependencies imported successfully')"
!python -m pytest -q


In [ ]:
!python -m src.download_data --output data/visd4sa
!python -m src.audit --data data/visd4sa

In [ ]:
!python -c "import py_vncorenlp; py_vncorenlp.download_model(save_dir='models/vncorenlp')"
!python -m src.prepare --data data/visd4sa --vncorenlp models/vncorenlp

## Nhãn overall

Thí nghiệm chính trong paper sử dụng nhãn Overall được suy ra từ
rating UIT-ViSFD, ghép với review UIT-ViSD4SA.

Dữ liệu đã xử lý được lưu tại `data/overall_visfd/`.
Không sử dụng `majority_only` để tái tạo kết quả trong paper.

In [ ]:
!python -m src.train_baseline --overall-source visfd --data data/overall_visfd --output outputs/baseline_visfd

In [ ]:
!python -m src.train_phobert --task overall --overall-source visfd --data data/overall_visfd --output outputs/phobert_visfd --fp16

In [ ]:
!python -m src.train_phobert --task aspect --fp16

In [ ]:
!python -m src.evaluate_model --checkpoint outputs/phobert/aspect/best --split test

In [ ]:
from src.inference import ABSAInference
model = ABSAInference()
print(model.predict_review('Máy chạy nhanh nhưng pin yếu'))

## Sau khi chạy
Giữ nguyên các file `outputs/**/metrics.json`, `outputs/**/best/`, `outputs/aspect_test_predictions.jsonl` và preprocessing report. Không báo cáo bất cứ số liệu nào chưa chạy. Ghi lại GPU, thời gian train, số mẫu bị loại, label provenance và kiểm tra duplicates.